In [1]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install transformers datasets rouge-score sentencepiece accelerate -q

  Preparing metadata (setup.py) ... done


In [2]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else 'None'}")


Device: cuda
GPU: Tesla T4


In [48]:
# ── CELL 3: Load and validate data ────────────────────────────────────────────
#df = pd.read_csv("training_file.csv")
df = pd.read_csv("train_augmented_EDA.csv")
df = df[["chunk_text", "summary_text"]].dropna()
df = df[df["chunk_text"].str.strip() != ""]
df = df[df["summary_text"].str.strip() != ""]
df = df.reset_index(drop=True)

print(f"Total rows        : {len(df)}")
print(f"Avg input length  : {df['chunk_text'].str.split().str.len().mean():.0f} words")
print(f"Avg target length : {df['summary_text'].str.split().str.len().mean():.0f} words")
df.head(2)

Total rows        : 381
Avg input length  : 361 words
Avg target length : 221 words


,chunk_text,summary_text
0,Vivad se Vishwas I Relief for MSMEs feature . ...,Vivad se Vishwas I will provide relief to MSME...
1,20 crore of central l tax and to reduce the am...,The upper limit proposed GST amendments furthe...


In [49]:
 # ── CELL 4: Train / val / test split ─────────────────────────────────────────
#train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
#val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)
train_df=df.copy()
val_df=pd.read_csv("val.csv")
test_df=pd.read_csv("test.csv")
print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)}")

def to_hf_dataset(dataframe):
    return Dataset.from_dict({
        "input_text" : dataframe["chunk_text"].tolist(),
        "target_text": dataframe["summary_text"].tolist(),
    })

dataset = DatasetDict({
    "train"     : to_hf_dataset(train_df),
    "validation": to_hf_dataset(val_df),
    "test"      : to_hf_dataset(test_df),
})
print(dataset)


Train : 381
Val   : 16
Test  : 16
DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 381
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 16
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 16
    })
})


In [50]:
# ── CELL 5: Tokenizer + tokenization ─────────────────────────────────────────
MODEL_CKPT = "t5-small"

# T5 needs a task prefix — this tells the model what to do with the input
# It was trained with these prefixes so always include it
TASK_PREFIX = "summarize: "

MAX_INPUT_LEN  = 512   # T5-base max is 512 — unlike BART's 1024
MAX_TARGET_LEN = 200

tokenizer = T5Tokenizer.from_pretrained(MODEL_CKPT)

def preprocess(batch):
    # Prepend task prefix to every input
    inputs = [TASK_PREFIX + text for text in batch["input_text"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=["input_text", "target_text"],
)
print(tokenized)


Map:   0%|          | 0/381 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 381
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 16
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 16
    })
})


In [51]:
# ── CELL 6: Model ─────────────────────────────────────────────────────────────
model = T5ForConditionalGeneration.from_pretrained(MODEL_CKPT)
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable_params/1e6:.1f}M")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Total params    : 60.5M
Trainable params: 60.5M


In [52]:
# ── CELL 7: Data collator ─────────────────────────────────────────────────────
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

In [53]:
!pip install bert-score -q

from bert_score import score as bert_score_fn

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # If model returns logits (3D) instead of token ids (2D), take argmax
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    vocab_size = tokenizer.vocab_size

    # Clip + cast — fast tokenizer (Rust) overflows on int64 or out-of-range ids
    predictions = np.clip(predictions, 0, vocab_size - 1).astype(np.int32)
    labels      = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels      = np.clip(labels, 0, vocab_size - 1).astype(np.int32)

    # Decode row by row to isolate any remaining bad tokens
    decoded_preds, decoded_labels = [], []
    for pred_ids, label_ids in zip(predictions.tolist(), labels.tolist()):
        try:
            decoded_preds.append(tokenizer.decode(pred_ids, skip_special_tokens=True).strip())
        except Exception:
            decoded_preds.append("")
        try:
            decoded_labels.append(tokenizer.decode(label_ids, skip_special_tokens=True).strip())
        except Exception:
            decoded_labels.append("")

    # ── ROUGE ────────────────────────────────────────────────────────────────
    r1 = r2 = rl = 0.0
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = scorer.score(label, pred)
        r1 += scores["rouge1"].fmeasure
        r2 += scores["rouge2"].fmeasure
        rl += scores["rougeL"].fmeasure

    n = len(decoded_preds)

    # ── BERTScore ─────────────────────────────────────────────────────────────
    # distilbert-base-uncased — lightweight, fast on Colab T4
    # Returns Precision, Recall, F1 tensors per sample — we report mean F1
    P, R, F1 = bert_score_fn(
        decoded_preds,
        decoded_labels,
        lang="en",
        model_type="distilbert-base-uncased",
        device=device,
        verbose=False,
    )

    return {
        "rouge1"       : round(r1 / n, 4),
        "rouge2"       : round(r2 / n, 4),
        "rougeL"       : round(rl / n, 4),
        "bertscore_P"  : round(P.mean().item(), 4),
        "bertscore_R"  : round(R.mean().item(), 4),
        "bertscore_F1" : round(F1.mean().item(), 4),
    }

In [54]:
# ── CELL 9: Training arguments ────────────────────────────────────────────────
BATCH_SIZE = 8      # T5-base is lighter than BART — can push batch size higher
GRAD_ACCUM = 2      # effective batch = 8 × 2 = 16
LR         = 3e-4   # T5 typically needs a higher LR than BART
EPOCHS     = 15
OUTPUT_DIR = "./t5-budget-summarizer"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    fp16=True,
    dataloader_num_workers=2,

    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [55]:
# ── CELL 10: Trainer ──────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,          # transformers >= 4.46
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer.train()


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bertscore P,Bertscore R,Bertscore F1
1,No log,2.915217,0.364200,0.140200,0.231700,0.827500,0.770600,0.797400
2,No log,2.528570,0.437400,0.182200,0.284100,0.828300,0.802700,0.814800
3,8.531155,2.371904,0.456800,0.202700,0.308400,0.837000,0.810700,0.823500
4,8.531155,2.306155,0.481400,0.226300,0.314800,0.838500,0.822000,0.829900
5,5.795550,2.288983,0.517700,0.248300,0.348400,0.848100,0.837100,0.842400
6,5.795550,2.261209,0.487700,0.225100,0.318600,0.836000,0.828200,0.831900
7,5.078813,2.264506,0.517300,0.247400,0.338600,0.845500,0.839600,0.842200
8,5.078813,2.251687,0.504600,0.237000,0.321300,0.845600,0.831700,0.838400
9,4.628721,2.258344,0.510200,0.237000,0.329700,0.845700,0.836200,0.840800
10,4.628721,2.282683,0.511200,0.221000,0.327000,0.842500,0.835200,0.838700


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=240, training_loss=5.745568402608236, metrics={'train_runtime': 268.2768, 'train_samples_per_second': 21.303, 'train_steps_per_second': 1.342, 'total_flos': 515652263608320.0, 'train_loss': 5.745568402608236, 'epoch': 10.0})

In [56]:
# ── CELL 11: Evaluate on test set ─────────────────────────────────────────────
results = trainer.evaluate(tokenized["test"])
print("\nTest Set Results:")
for k, v in results.items():
    print(f"  {k:<30} {v}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Test Set Results:
  eval_loss                      2.0357871055603027
  eval_rouge1                    0.5415
  eval_rouge2                    0.2695
  eval_rougeL                    0.3813
  eval_bertscore_P               0.8572
  eval_bertscore_R               0.8371
  eval_bertscore_F1              0.8466
  eval_runtime                   14.9541
  eval_samples_per_second        1.07
  eval_steps_per_second          0.134
  epoch                          10.0


In [57]:
# ── CELL 12: Save model ───────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR + "/final")
tokenizer.save_pretrained(OUTPUT_DIR + "/final")
print(f"Model saved to {OUTPUT_DIR}/final")

# Persist to Drive:
from google.colab import drive
drive.mount('/content/drive')
!cp -r {OUTPUT_DIR}/final /content/drive/MyDrive/t5-budget-summarizer

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./t5-budget-summarizer/final
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
output_file_path = f"{OUTPUT_DIR}/final/evaluation_results.txt"
with open(output_file_path, "w") as f:
    f.write("Test Set Results:\n")
    for k, v in results.items():
        f.write(f"  {k:<30} {v}\n")
print(f"Evaluation results saved to {output_file_path}")

Evaluation results saved to ./t5-budget-summarizer/final/evaluation_results.txt


In [60]:
# ── CELL 13: Inference ────────────────────────────────────────────────────────
def summarize(text: str, max_length: int = 500, min_length: int = 50) -> str:
    # Must include the same task prefix used during training
    input_text = TASK_PREFIX + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LEN,
        truncation=True,
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=max_length,
        min_length=min_length,
        length_penalty=3.0,
        early_stopping=False,
        no_repeat_ngram_size=3,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


sample_text = test_df.iloc[8]["chunk_text"]
actual      = test_df.iloc[8]["summary_text"]
predicted   = summarize(sample_text)

print(f"\nINPUT    : {sample_text[:300]}...\n")
print(f"ACTUAL   : {actual}\n")
print(f"PREDICTED: {predicted}")


INPUT    : Digital Ecosystem for Skilling and Livelihood the DESH-Stack e- portal will be launched. This aims to empower citizens to skill, reskill or upskill through on-line training. It will also provide API-based trusted skill credentials, payment and discovery layers to find relevant jobs and entrepreneuri...

ACTUAL   : The Budget proposed creation of a Digital Ecosystem for Skilling and Livelihood (DESH-Stack e-portal) to support online skilling, reskilling, and upskilling through digital training, API-based skill credentials, payment systems, and job and entrepreneurship discovery services. To encourage emerging technologies, startups would be promoted under the Drone Shakti initiative for drone applications and Drone-as-a-Service (DrAAS), along with introduction of drone-related skill courses in selected ITIs across all states. Addressing pandemic-related learning losses, especially among rural, SC/ST, and economically weaker students in government schools, the PM eVIDYA one c